# Aria Deployment & DevOps Guide

**Complete guide for deploying Aria to Azure, managing infrastructure, monitoring, and operational runbooks.**

Learn deployment strategies, infrastructure setup, monitoring, and production operations.


## Deployment Architecture

### Components to Deploy

```
┌─────────────────────────────────────────────────────────┐
│                  PRODUCTION ENVIRONMENT                  │
├─────────────────────────────────────────────────────────┤
│                                                           │
│  Azure Static Web App          Azure Functions            │
│  ├─ Aria Web UI                ├─ /api/chat              │
│  ├─ Chat Interface             ├─ /api/tts               │
│  └─ Dashboard                  ├─ /api/quantum/*         │
│                                └─ /api/ai/status        │
│                                                           │
│  Azure App Service             Azure Cosmos DB            │
│  └─ Aria Web Server (8080)     └─ Chat memory            │
│                                                           │
│  Azure SQL / PostgreSQL        Azure Storage              │
│  └─ Chat history              └─ Models/artifacts       │
│                                                           │
└─────────────────────────────────────────────────────────┘
```

---

## Local Development Setup

### Prerequisites

```bash
# Install required tools
brew install python@3.12 node npm azure-cli func

# Clone repository
git clone https://github.com/Bryan-Roe/Aria.git
cd Aria

# Create virtual environment
python3.12 -m venv .venv
source .venv/bin/activate

# Install dependencies
pip install -r requirements-dev.txt
npm install -g azure-functions-core-tools@4

# Set up local configuration
cp local.settings.json.example local.settings.json
# Edit local.settings.json with your settings
```

### Starting Local Environment

```bash
# Terminal 1: Azure Functions (port 7071)
func host start --port 7071

# Terminal 2: Aria Web Server (port 8080)
cd apps/aria
python server.py --port 8080

# Terminal 3: Storage Emulator (optional)
azurite --location ./local_data

# Test endpoints
curl http://localhost:7071/api/ai/status
curl http://localhost:8080/api/aria/state
```

---

## Azure Deployment

### Step 1: Create Azure Resources

```bash
#!/bin/bash

# Set variables
RESOURCE_GROUP="aria-rg"
LOCATION="eastus"
APP_NAME="aria-app"
STORAGE_NAME="ariastorage$(date +%s)"

# Create resource group
az group create --name $RESOURCE_GROUP --location $LOCATION

# Create storage account
az storage account create \
  --resource-group $RESOURCE_GROUP \
  --name $STORAGE_NAME \
  --sku Standard_LRS

# Create Azure Functions app
az functionapp create \
  --resource-group $RESOURCE_GROUP \
  --consumption-plan-location $LOCATION \
  --runtime python \
  --runtime-version 3.12 \
  --functions-version 4 \
  --name $APP_NAME \
  --storage-account $STORAGE_NAME

# Create App Service for Aria web server
az appservice plan create \
  --name $APP_NAME-plan \
  --resource-group $RESOURCE_GROUP \
  --sku B1 \
  --is-linux

az webapp create \
  --resource-group $RESOURCE_GROUP \
  --plan $APP_NAME-plan \
  --name $APP_NAME-web \
  --runtime "PYTHON:3.12"
```

### Step 2: Configure Environment Variables

```bash
# Set Azure Functions environment variables
az functionapp config appsettings set \
  --name $APP_NAME \
  --resource-group $RESOURCE_GROUP \
  --settings \
    "AZURE_OPENAI_API_KEY=$AZURE_OPENAI_API_KEY" \
    "AZURE_OPENAI_ENDPOINT=$AZURE_OPENAI_ENDPOINT" \
    "QAI_DB_CONN=$DB_CONNECTION_STRING" \
    "COSMOS_ENDPOINT=$COSMOS_ENDPOINT" \
    "APPLICATIONINSIGHTS_CONNECTION_STRING=$INSIGHTS_KEY"

# Verify settings
az functionapp config appsettings list \
  --name $APP_NAME \
  --resource-group $RESOURCE_GROUP
```

### Step 3: Deploy Code

```bash
# Deploy Azure Functions
func azure functionapp publish $APP_NAME

# Deploy Aria Web Server (via App Service)
cd apps/aria
zip -r deployment.zip . -x "node_modules/*" "__pycache__/*"

az webapp deployment source config-zip \
  --resource-group $RESOURCE_GROUP \
  --name $APP_NAME-web \
  --src-path deployment.zip

# Start the app
az webapp start \
  --resource-group $RESOURCE_GROUP \
  --name $APP_NAME-web

# Verify deployment
curl https://$APP_NAME.azurewebsites.net/api/ai/status
```

---

## Monitoring & Observability

### Application Insights Setup

```bash
# Create Application Insights
az monitor app-insights component create \
  --app aria-insights \
  --location $LOCATION \
  --resource-group $RESOURCE_GROUP

# Get connection string
az monitor app-insights component show \
  --app aria-insights \
  --resource-group $RESOURCE_GROUP \
  --query connectionString
```

### Monitoring Queries (Kusto Query Language)

```kusto
# Error rate over time
requests
| where cloud_RoleName == "aria-api"
| where resultCode >= 400
| summarize ErrorCount=count() by bin(timestamp, 5m)
| render timechart

# Slow endpoint detection
requests
| where cloud_RoleName == "aria-api"
| where duration > 5000
| summarize SlowCount=count(), AvgDuration=avg(duration) by name
| order by SlowCount desc

# Exception analysis
exceptions
| where cloud_RoleName == "aria-api"
| summarize count() by type, outerMessage
| order by count_ desc
```

### Health Check Alerts

```bash
# Create alert for high error rate
az monitor metrics alert create \
  --name "Aria-HighErrorRate" \
  --resource-group $RESOURCE_GROUP \
  --scopes /subscriptions/.../providers/microsoft.insights/components/aria-insights \
  --condition "avg requests/failed > 10" \
  --evaluation-frequency 1m \
  --window-size 5m \
  --severity 2
```

---

## Scaling & Performance

### Auto-Scaling Configuration

```bash
# Create autoscale setting for App Service
az monitor autoscale create \
  --resource-group $RESOURCE_GROUP \
  --resource-name $APP_NAME-web \
  --resource-type "Microsoft.Web/serverfarms" \
  --min-count 1 \
  --max-count 5 \
  --count 1

# Add scale-up rule (CPU > 70%)
az monitor autoscale rule create \
  --autoscale-name $APP_NAME-web-autoscale \
  --resource-group $RESOURCE_GROUP \
  --metric-name "CpuPercentage" \
  --metric-resource-type "Microsoft.Web/serverfarms" \
  --metric-statistic Average \
  --threshold 70 \
  --operator GreaterThan \
  --scale-out-cooldown 5m \
  --scale-out-count 2
```

### Load Testing

```bash
# Create load test
az load test create \
  --load-test-resource aria-load-test \
  --resource-group $RESOURCE_GROUP \
  --display-name "Aria API Load Test" \
  --description "Test API endpoints under load"

# Run load test with Apache JMeter script
az load test file upload \
  --load-test-resource aria-load-test \
  --resource-group $RESOURCE_GROUP \
  --file-type "JMeter Script" \
  --path "./tests/load/api_load_test.jmx"

az load test run create \
  --load-test-resource aria-load-test \
  --resource-group $RESOURCE_GROUP \
  --display-name "Peak Load Test"
```

---

## Backup & Disaster Recovery

### Database Backup Strategy

```bash
# Enable automated backups for SQL Database
az sql db update \
  --server $SERVER_NAME \
  --name aria_db \
  --resource-group $RESOURCE_GROUP \
  --backup-storage-redundancy Geo

# Create manual backup
az sql db copy \
  --dest-server $SERVER_NAME \
  --name aria_db \
  --dest-name aria_db_backup_$(date +%Y%m%d) \
  --resource-group $RESOURCE_GROUP

# Configure retention (35 days)
az sql db short-term-retention-policy update \
  --server $SERVER_NAME \
  --database aria_db \
  --retention-days 35 \
  --resource-group $RESOURCE_GROUP
```

### Disaster Recovery Runbook

```bash
#!/bin/bash
# Restore from backup

# 1. Check backup status
az sql db list --server $SERVER_NAME --resource-group $RESOURCE_GROUP

# 2. Restore from point-in-time
RESTORE_TIME=$(date -u -d '1 hour ago' +%Y-%m-%dT%H:%M:%SZ)

az sql db restore \
  --server $SERVER_NAME \
  --name aria_db \
  --dest-name aria_db_restored \
  --resource-group $RESOURCE_GROUP \
  --time $RESTORE_TIME

# 3. Verify restore
az sql db show \
  --server $SERVER_NAME \
  --name aria_db_restored \
  --resource-group $RESOURCE_GROUP

# 4. If good, swap connection strings
# Update app settings to point to restored DB
```

---

## CI/CD Pipeline

### GitHub Actions Deployment Workflow

```yaml
name: Deploy to Azure

on:
    push:
        branches: [main]
    pull_request:
        branches: [main]

jobs:
    test:
        runs-on: ubuntu-latest
        steps:
            - uses: actions/checkout@v3
            - name: Set up Python
              uses: actions/setup-python@v4
              with:
                  python-version: "3.12"
            - name: Run tests
              run: |
                  pip install -r requirements-dev.txt
                  python scripts/test_runner.py --unit

    deploy:
        needs: test
        if: github.ref == 'refs/heads/main'
        runs-on: ubuntu-latest
        steps:
            - uses: actions/checkout@v3

            - name: Log in to Azure
              uses: azure/login@v1
              with:
                  creds: ${{ secrets.AZURE_CREDENTIALS }}

            - name: Deploy to Functions
              run: |
                  npm install -g azure-functions-core-tools@4
                  func azure functionapp publish ${{ secrets.FUNCTION_APP_NAME }}

            - name: Deploy to App Service
              uses: azure/webapps-deploy@v2
              with:
                  app-name: ${{ secrets.APP_SERVICE_NAME }}
                  package: apps/aria

            - name: Run smoke tests
              run: |
                  curl https://${{ secrets.APP_SERVICE_NAME }}.azurewebsites.net/api/ai/status
                  curl https://${{ secrets.FUNCTION_APP_NAME }}.azurewebsites.net/api/ai/status
```

---

## Production Runbooks

### Runbook: Emergency Stop

```bash
#!/bin/bash
# Stop all components in emergency

# Stop Functions
az functionapp stop --resource-group $RESOURCE_GROUP --name $APP_NAME

# Stop App Service
az webapp stop --resource-group $RESOURCE_GROUP --name $APP_NAME-web

# Scale to zero (if on consumption plan)
# az monitor autoscale update --auto-scale-name $APP_NAME-autoscale \
#   --resource-group $RESOURCE_GROUP \
#   --min-count 0

echo "Aria services stopped (emergency)"
```

### Runbook: Rollback Failed Deployment

```bash
#!/bin/bash
# Rollback to previous deployment

# Get deployment history
az webapp deployment list \
  --resource-group $RESOURCE_GROUP \
  --name $APP_NAME-web

# Swap to previous slot
az webapp deployment slot swap \
  --resource-group $RESOURCE_GROUP \
  --name $APP_NAME-web \
  --slot staging

# Verify
curl https://$APP_NAME-web.azurewebsites.net/api/ai/status
echo "Rollback complete"
```

### Runbook: Database Performance Issue

```bash
#!/bin/bash
# Diagnose and fix database performance

# Check DTU usage
az sql db show --server $SERVER_NAME --name aria_db --resource-group $RESOURCE_GROUP

# Kill long-running queries
az sql db audit-policy show \
  --server $SERVER_NAME \
  --database aria_db \
  --resource-group $RESOURCE_GROUP

# Scale up if needed
az sql db update \
  --server $SERVER_NAME \
  --name aria_db \
  --service-objective S2 \
  --resource-group $RESOURCE_GROUP

# Monitor improvement
# ... check metrics in Azure Portal
```


## Deployment Checklist

### Pre-Deployment

- [ ] All tests pass: `python scripts/test_runner.py --all`
- [ ] Code reviewed and approved
- [ ] No hardcoded secrets in code
- [ ] Environment variables documented
- [ ] Database migration script tested
- [ ] Backup created before deployment

### During Deployment

- [ ] Deploy to staging environment first
- [ ] Run smoke tests: `scripts/smoke_tests.sh`
- [ ] Monitor error rate (should be <1%)
- [ ] Monitor response times (should be <500ms)
- [ ] Have rollback plan ready

### Post-Deployment

- [ ] Verify all endpoints responding
- [ ] Check error logs for issues
- [ ] Monitor resource usage
- [ ] Confirm feature working end-to-end
- [ ] Update deployment status page

### Rollback Criteria

- Error rate >5%
- Response time >2s
- Database connection issues
- Critical functionality broken
- Memory usage >90%

---

## Quick Deployment Commands

```bash
# Verify deployment readiness
python scripts/fast_validate.py
make lint
python scripts/test_runner.py --unit

# Deploy to staging
func azure functionapp publish aria-app-staging

# Deploy to production
func azure functionapp publish aria-app

# Check deployment status
az functionapp show --name aria-app --resource-group aria-rg

# View live logs
az webapp log tail --resource-group aria-rg --name aria-web

# Rollback to previous version
az webapp deployment slot swap --resource-group aria-rg --name aria-web --slot staging
```
